# TabPFN-TS-3 — Price
reproduces main results: MAE, RMSE, aQL.

In [ ]:
# !pip install -r requirements/tabpfn.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"price"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode
from preprocessing import load_data, describe_data, SITES, CONTEXT_LENGTH, PREDICTION_LENGTH, FUTURE_COVARIATES, PAST_COVARIATES
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=OUTPUT_QUANTILES
OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data=load_data(ROOT/"data")
print(describe_data(data))


## GEFCom2014-P


In [ ]:
# GEFCom2014-P (cleaned input; no extra cleaning)
from gefcom import load_gefcom_price, price_context_and_days, evaluate_gefcom_price, PRICE_COVARIATES
gef_price=load_gefcom_price(ROOT/"data"/"gefcom_price_data.csv")
gef_context, gef_test, gef_days=price_context_and_days(gef_price)
GEF_OUTPUT=ROOT/"outputs"/"predictions"; GEF_OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
GEF_VARIANT="with_covariates"
gef_pipeline=TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)
rolling=gef_context.copy(); frames=[]
for _,future in tqdm(gef_days):
    context=rolling.sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
    if GEF_VARIANT=="with_covariates": pred=gef_pipeline.predict_df(context,future_df=future[["item_id","timestamp",*PRICE_COVARIATES]],prediction_length=24)
    else: pred=gef_pipeline.predict_df(context[["item_id","timestamp","target"]],prediction_length=24)
    frames.append(pred); rolling=pd.concat([rolling,future],ignore_index=True)
gef_result=pd.concat(frames,ignore_index=True).merge(gef_test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")
if "prediction" not in gef_result and "0.5" in gef_result: gef_result["prediction"]=gef_result["0.5"]
gef_result.to_csv(GEF_OUTPUT/f"tabpfn_gefcom_price_{GEF_VARIANT}.csv",index=False)
gef_summary=evaluate_gefcom_price(gef_result,QUANTILES); display(gef_summary[["task_date","raw_aql","normalized_aql"]])
gef_summary.to_csv(GEF_OUTPUT/f"tabpfn_gefcom_price_summary.csv",index=False)


## GridStatus2025


In [ ]:
VARIANT="with_covariates"
FUTURE_COLS=FUTURE_COVARIATES
pipeline=TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)


In [ ]:
rolling={s:data.context_by_site[s].copy() for s in data.context_by_site}
frames=[]
for i in tqdm(range(len(next(iter(data.daily_by_site.values()))))):
    day={s:data.daily_by_site[s][i][1].sort_values("timestamp").iloc[:PREDICTION_LENGTH].copy() for s in data.daily_by_site}
    for site,future in day.items():
        context=rolling[site].sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
        if VARIANT=="with_covariates":
            pred=pipeline.predict_df(context,future_df=future[["item_id","timestamp",*FUTURE_COLS]],quantiles=MODEL_QUANTILES)
        else:
            pred=pipeline.predict_df(context[["item_id","timestamp","target"]],prediction_length=PREDICTION_LENGTH,quantiles=MODEL_QUANTILES)
        frames.append(pred)
    for site in day: rolling[site]=pd.concat([rolling[site],day[site]],ignore_index=True).sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
result=pd.concat(frames,ignore_index=True)
if "target" in result.columns: result=result.rename(columns={"target":"prediction"})
if "prediction" not in result.columns: result["prediction"]=result["0.5"]
result=result.merge(data.test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR/f"tabpfn_ts_{VARIANT}.csv",index=False)
print_metrics(result,QUANTILES,label=f"tabpfn_ts_{VARIANT}")
